### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
# Read all pdf's file

def process_all_pdfs(pdf_directory):
    
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"    Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [4]:
all_pdf_docs = process_all_pdfs("../data/")

Found 5 PDF files to process.

 Processing: AI Learning Resources (Beyond the Basics).pdf
    Loaded 2 pages

 Processing: AI_ML_Master_Roadmap_2026_Outline.pdf
    Loaded 2 pages

 Processing: 3_Month_AI_ML_Roadmap.pdf
    Loaded 1 pages

 Processing: Design_Patterns_Interview_Revision.pdf
    Loaded 1 pages

 Processing: Week 1.pdf
    Loaded 57 pages

Total documents loaded: 63


In [5]:
# Text Splitting - Chunking

def split_documents(documents, chunk_size=500, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks = split_documents(all_pdf_docs)

Split 63 documents into 72 chunks

Example chunk: 
Content: 🚀  AI  Learning  Resources  (Beyond  the  
Basics)
 
If  you're  serious  about  AI,  here  are  some  of  the  resources  that  will  make  you  stand  out.  
🧠  Learn  how  great  engineers  think  ...
Metadata: {'producer': 'Skia/PDF m152 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': '🚀 AI Learning Resources (Beyond the Basics)', 'source': '../data/pdf/AI Learning Resources (Beyond the Basics).pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'AI Learning Resources (Beyond the Basics).pdf', 'file_type': 'pdf'}


#### Embedding and Vector DB

In [ ]:
import uuid
import chromadb
import warnings
import numpy as np
warnings.filterwarnings('ignore')
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    
    def __init__(self, model_name: str="all-MiniLM-L6-v2"): 
        self.model_name = model_name
        self.model = None
        self._load_model()
            
    def _load_model(self):
        
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully. Embedded Dimension: {self.model.get_embedding_dimension()}")
            
        except Exception as e:
            print(f"Error loading model {e}")
            raise
        
    def generate_embedding(self, text: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embedding for {len(text)} texts")
        embedding = self.model.encode(text, show_progress_bar=True)
        print(f"Generating embedding with shape: {len(embedding.shape)}")
        
        return embedding 

In [9]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6696.74it/s]


Model loaded Successfully. Embedded Dimension: 384


#### Vector Store

In [ ]:
class VectorStore:
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        
        try: 
            # Create Persistant client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or Create Collections
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata= {
                    "Description": "PDF documents embedding for RAG"
                }
            )
            
            print(f"Vector Store Initialized. Collection: {self.collection_name}")
            print(f"Existing Documents in collections: {self.collection.count()}")
        
        except Exception as e:
            
            print(f"Error Initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of Documents must match the number of embeddings")

        print(f"Adding {len(documents)} documents to Vector Store")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            
            # Generating IDs
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Generating Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Documents Content
            documents_text.append(doc.page_content)
            
            # Embeddings
            embeddings_list.append(embedding.tolist())
            
        try: 
            self.collection.add(
                ids = ids,
                metadatas= metadatas,
                embeddings= embeddings_list,
                documents= documents_text
            )
            
            print(f"Added {len(documents)} documents to Vector Store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to Vector Store {e}")
            raise

In [11]:
vector_store = VectorStore()
vector_store

Vector Store Initialized. Collection: pdf_documents
Existing Documents in collections: 144


In [12]:
# Converting text to embeddings
texts = [doc.page_content for doc in chunks]

# Generating Embeddings
embedding = embedding_manager.generate_embedding(texts)

# Storing in Vector Database
vector_store.add_documents(chunks, embedding)

Generating embedding for 72 texts


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.25it/s]

Generating embedding with shape: 2
Adding 72 documents to Vector Store
Added 72 documents to Vector Store
Total documents in collection: 216


### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"\nRetrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score Threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding,
                n_results= top_k
            )
        
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'distance': distance,
                            'rank': i + 1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")
                
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval {e}")
            return []

In [14]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [ ]:
rag_retriever.retrieve("What are types of Supervised Machine Learning Algorithms?")


Retrieving documents for query: 'What are types of Supervised Machine Learning Algorithms?'
Top k: 5, Score Threshold: 0.0
Generating embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.04it/s]

Generating embedding with shape: 2
Retrieved 5 documents (after filtering)


[{'id': 'doc_4ff8accd_26',
  'content': 'Types of Machine Learning',
  'metadata': {'moddate': '2025-10-03T08:10:19+05:00',
   'doc_index': 26,
   'source_file': 'Week 1.pdf',
   'title': 'Microsoft PowerPoint - Week 1  -  Compatibility Mode',
   'content_length': 25,
   'total_pages': 57,
   'page': 14,
   'creator': 'PyPDF',
   'file_type': 'pdf',
   'creationdate': '2025-10-03T08:10:19+05:00',
   'page_label': '15',
   'producer': 'Microsoft: Print To PDF',
   'source': '../data/pdf/Week 1.pdf',
   'author': 'Irfan'},
  'distance': 0.27195942401885986,
  'rank': 1},
 {'id': 'doc_47077c7b_26',
  'content': 'Types of Machine Learning',
  'metadata': {'source_file': 'Week 1.pdf',
   'source': '../data/pdf/Week 1.pdf',
   'title': 'Microsoft PowerPoint - Week 1  -  Compatibility Mode',
   'doc_index': 26,
   'content_length': 25,
   'creationdate': '2025-10-03T08:10:19+05:00',
   'author': 'Irfan',
   'moddate': '2025-10-03T08:10:19+05:00',
   'creator': 'PyPDF',
   'page_label': '15',


### Integrating VectorDB Context Pipeline with LLM output

In [17]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [24]:
llm = ChatGroq(api_key=groq_api_key, model_name= "llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    result = retriever.retrieve(query, top_k=top_k)
    
    context="\n\n".join([doc['content'] for doc in result]) if result else ""
    
    if not context:
        return "No relevent context found to answer the question"
    
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}
        
        Question:
        {query}
        
        Answers:"""
        
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [25]:
answer = rag_simple("What are types of Supervised Machine Learning Algorithms?", rag_retriever, llm)
print(answer)


Retrieving documents for query: 'What are types of Supervised Machine Learning Algorithms?'
Top k: 3, Score Threshold: 0.0
Generating embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.71it/s]

Generating embedding with shape: 2
Retrieved 3 documents (after filtering)


**Types of Supervised Machine Learning Algorithms:**

1. **Linear Regression**: Predicts continuous outcomes based on one or more predictor variables.
2. **Logistic Regression**: Predicts binary outcomes (0/1, yes/no) based on one or more predictor variables.
3. **Decision Trees**: Classify data by creating a tree-like model of decisions.
4. **Random Forest**: Ensemble learning method that combines multiple decision trees.
5. **Support Vector Machines (SVMs)**: Find the hyperplane that maximally separates classes in feature space.
6. **K-Nearest Neighbors (KNN)**: Classify data based on the majority vote of its k nearest neighbors.
7. **Naive Bayes**: Classify data based on Bayes' theorem with a naive assumption of independence between features.
8. **Gradient Boosting**: Ensemble learning method that combines multiple weak models to create a strong predictive model.


### Enhanced RAG Pipeline Features